In [1]:
from sklearn.datasets import (load_iris, load_breast_cancer, load_digits, load_wine,
                              fetch_covtype, fetch_kddcup99, fetch_lfw_pairs, fetch_lfw_people,
                              fetch_olivetti_faces, fetch_rcv1)
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder

import pandas as pd


from export_sklearn_data.find_interesting_vectors import find_vectors
from dt_explanation import write_dttxt_file

In [2]:
# Détection automatique des colonnes catégorielles
categorical_features = make_column_selector(dtype_include=['object', 'category'])

# Transformation : OneHotEncoder uniquement sur les colonnes catégorielles
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough',  # Garde les autres colonnes inchangées
    verbose_feature_names_out=False
)

In [3]:
# data_loading_functions = [fetch_kddcup99, fetch_lfw_pairs, fetch_lfw_people, fetch_olivetti_faces, fetch_rcv1]
f = fetch_covtype
output_filename = "dttxt_files/kddcup99.txt"

In [11]:
raw_data = f(as_frame=True)
raw_data = f()

In [12]:
rawX = raw_data.data

In [13]:
rawX[0]

array([2.596e+03, 5.100e+01, 3.000e+00, 2.580e+02, 0.000e+00, 5.100e+02,
       2.210e+02, 2.320e+02, 1.480e+02, 6.279e+03, 1.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00])

### transform non float features

In [ ]:
raw_X = raw_data['data'].copy()
raw_y = raw_data['target']

In [11]:
# Colonnes réellement catégorielles dans KDD Cup 99
categorical_cols = ['protocol_type', 'service', 'flag']

# Toutes les autres colonnes doivent être numériques
numeric_cols = [c for c in raw_X.columns if c not in categorical_cols]

In [12]:
# Conversion : object -> numérique (gère aussi les bytes type b'0')
for col in numeric_cols:
    raw_X[col] = pd.to_numeric(raw_X[col], errors='coerce')

In [13]:
# Les colonnes catégorielles restent en str si elles sont en bytes
for col in categorical_cols:
    if isinstance(raw_X[col].iloc[0], bytes):
        raw_X[col] = raw_X[col].str.decode('utf-8')

In [20]:
X = preprocessor.fit_transform(raw_X)

feature_names = preprocessor.get_feature_names_out()
print(f"nb_features : {len(feature_names)}")
print(list(feature_names))

nb_features : 118
['protocol_type_icmp', 'protocol_type_tcp', 'protocol_type_udp', 'service_IRC', 'service_X11', 'service_Z39_50', 'service_auth', 'service_bgp', 'service_courier', 'service_csnet_ns', 'service_ctf', 'service_daytime', 'service_discard', 'service_domain', 'service_domain_u', 'service_echo', 'service_eco_i', 'service_ecr_i', 'service_efs', 'service_exec', 'service_finger', 'service_ftp', 'service_ftp_data', 'service_gopher', 'service_hostnames', 'service_http', 'service_http_443', 'service_imap4', 'service_iso_tsap', 'service_klogin', 'service_kshell', 'service_ldap', 'service_link', 'service_login', 'service_mtp', 'service_name', 'service_netbios_dgm', 'service_netbios_ns', 'service_netbios_ssn', 'service_netstat', 'service_nnsp', 'service_nntp', 'service_ntp_u', 'service_other', 'service_pm_dump', 'service_pop_2', 'service_pop_3', 'service_printer', 'service_private', 'service_red_i', 'service_remote_job', 'service_rje', 'service_shell', 'service_smtp', 'service_sql_ne

In [16]:
y = raw_y.str.decode('utf-8') if isinstance(raw_y.iloc[0], bytes) else raw_y

## Treat data :

In [17]:
## Split dataset into training set and test set
X_train, X_test, y_train, _ = train_test_split(X, y, test_size=0.1, random_state=1) # 90% training and 10% test

In [18]:
## create and fit tree
dt = tree.DecisionTreeClassifier(random_state=0, max_depth=None)
dt = dt.fit(X_train, y_train)

In [22]:
## create vectors
# vs = find_vectors(dt, X.shape[0])
vs = [X_test[0]]

In [23]:
write_dttxt_file(dt, vs, feature_names=feature_names, data_filename=output_filename)